# Explainability Evaluation - Stability

## Testing the Attention based Explainability and SHAP on Stability

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
/content/bachelor-thesis
Already up to date.
/content/bachelor-thesis
Repo exists: True


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import shap
import torch
from torch.utils.data import DataLoader, TensorDataset

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.anoformer_attention import AnoFormerExplainability
from explainability.tranad_attention import TranADExplainability
from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel
from explainability.evaluation import StabilityEvaluation
from explainability.shap_wrappers import AnoFormerAnomalyScoreWrapper, TranADAnomalyScoreWrapper


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [4]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

SHAP_RESULTS_DIR = RESULTS_DIR / "shap"

STABILITY_DIR = RESULTS_DIR / "evaluation" / "stability"
STABILITY_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [6]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
# We need the Train dataset because the SHAP background is from the training set. We also need the mean and std of the training set to normalize the test set.
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)

# We need the Test dataset because the chosen samples for evaluation are from the test set
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [7]:
def make_single_ecg_loader(ecg):
    if isinstance(ecg, np.ndarray):
        ecg = torch.tensor(ecg, dtype=torch.float32)

    if ecg.ndim == 2:
        ecg = ecg.unsqueeze(0)

    dummy_label = torch.zeros(ecg.shape[0], dtype=torch.long)
    dataset = TensorDataset(ecg, dummy_label)

    return DataLoader(dataset, batch_size=1, shuffle=False)

In [8]:
# Load previosly chosen ECGs for Explainability and Evaluation

SharedSamples = np.load(
    RESULTS_DIR
    / "evaluation"
    / "shared_explanation_samples.npz"
)

NormalIndex = int(SharedSamples["normal_index"])
AnomalousIndex = int(SharedSamples["anomalous_index"])

NormalECG, NormalLabel = test_dataset[NormalIndex]
AnomalousECG, AnomalousLabel = test_dataset[AnomalousIndex]

NormalECG = NormalECG.unsqueeze(0)
AnomalousECG = AnomalousECG.unsqueeze(0)

print("Index of Normal ECG:", NormalIndex, "Label:", int(NormalLabel))
print("Index of Anomalous ECG:", AnomalousIndex, "Label:", int(AnomalousLabel))

print("Normal ECG shape:", NormalECG.shape)
print("Anomalous ECG shape:", AnomalousECG.shape)

Index of Normal ECG: 583 Label: 0
Index of Anomalous ECG: 495 Label: 1
Normal ECG shape: torch.Size([1, 1000, 12])
Anomalous ECG shape: torch.Size([1, 1000, 12])


In [ ]:
# Load previously calculated Anomaly Thresholds
# these were calculated based on the evaluation set

filepath = MODEL_DIR / "chosen_thresholds.json"

with open(filepath, "r") as f:
    thresholds = json.load(f)

AnoFormerChosenThreshold = thresholds["anoformer"]
TranADChosenThreshold = thresholds["tranad"]

In [9]:
# Create the Evaluator

StabilityEvaluator = StabilityEvaluation(
    noise_ratio=0.01,
    epsilon=1e-8
)

# AnoFormer Evaluation

In [ ]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)


Modell geladen von: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [11]:
def anoformer_score_function(ecg):
    """
    Return the AnoFormer anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = AnoFormer.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=AnoFormerChosenThreshold,
        NumMaskRuns=5,
        Seed=RANDOM_SEED,
    )

    return scores[0]

### Attention Evaluation

In [ ]:
# A helper function to get the attention importance values for a given ECG
# This is needed because we neeed to pass multiple ecgs

def anoformer_attention_importance_function(ecg):
    return AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        BlockIndex=0,
        HeadIndex=None,
    )

In [10]:
# to check whether the noisey ECG still has the same prediction, because we only use these

def get_prediction(score, threshold):
    return int(score >= threshold)

In [11]:
# This function collects valid noisy variants of the given ECG, ensuring that the prediction remains the same as the original ECG. It returns the noisy importance values and their corresponding scores.
# it loops until it collects the required number of valid noisy variants

def collect_valid_noisy_importances(
    ecg,
    score_function,
    explanation_function,
    threshold,
    stability_evaluator,
    number_variations=10,
):
    original_score = score_function(ecg)
    original_prediction = get_prediction(original_score, threshold)

    noisy_importances = []
    noisy_scores = []

    attempts = 0
    max_attempts = number_variations * 10

    while len(noisy_importances) < number_variations and attempts < max_attempts:
        attempts += 1

        noisy_ecg = stability_evaluator.add_gaussian_noise(ecg)

        noisy_score = score_function(noisy_ecg)
        noisy_prediction = get_prediction(noisy_score, threshold)

        if noisy_prediction != original_prediction:
            continue

        noisy_importance = explanation_function(noisy_ecg)

        noisy_importances.append(noisy_importance)
        noisy_scores.append(noisy_score)

    if len(noisy_importances) < number_variations:
        print(
            f"Warning: Only {len(noisy_importances)} valid noisy variants "
            f"were generated."
        )

    return noisy_importances, noisy_scores

In [15]:
# Extract the original attention importance values for the normal and anomalous ECGs

NormalAttentionImportance = anoformer_attention_importance_function(NormalECG)
AnomalousAttentionImportance = anoformer_attention_importance_function(AnomalousECG)

In [16]:
# Generate valid noisy variants of the normal ECG and collect their attention importance values and scores

NormalNoisyAttentionImportances, NormalNoisyScores = collect_valid_noisy_importances(
    ecg=NormalECG,
    score_function=anoformer_score_function,
    explanation_function=anoformer_attention_importance_function,
    threshold=AnoFormerChosenThreshold,
    stability_evaluator=StabilityEvaluator,
    number_variations=10,
)

AnomalousNoisyAttentionImportances, AnomalousNoisyScores = collect_valid_noisy_importances(
    ecg=AnomalousECG,
    score_function=anoformer_score_function,
    explanation_function=anoformer_attention_importance_function,
    threshold=AnoFormerChosenThreshold,
    stability_evaluator=StabilityEvaluator,
    number_variations=10,
)

In [17]:
# Evaluate
AnoFormerAttentionNormalStability = StabilityEvaluator.evaluate(
    original_importance=NormalAttentionImportance,
    noisy_importance_values=NormalNoisyAttentionImportances,
)

AnoFormerAttentionAnomalousStability = StabilityEvaluator.evaluate(
    original_importance=AnomalousAttentionImportance,
    noisy_importance_values=AnomalousNoisyAttentionImportances,
)

In [18]:
# Print the Results
AnoFormerAttentionStabilityResults = pd.DataFrame([
    {
        "Model": "AnoFormer",
        "Explanation": "Attention",
        "Sample": "Normal",
        "Stability Score": AnoFormerAttentionNormalStability["stability_score"],
        "Stability Std": AnoFormerAttentionNormalStability["stability_std"],
        "Valid Variations": len(NormalNoisyAttentionImportances),
    },
    {
        "Model": "AnoFormer",
        "Explanation": "Attention",
        "Sample": "Anomalous",
        "Stability Score": AnoFormerAttentionAnomalousStability["stability_score"],
        "Stability Std": AnoFormerAttentionAnomalousStability["stability_std"],
        "Valid Variations": len(AnomalousNoisyAttentionImportances),
    },
])

AnoFormerAttentionStabilityResults

,Model,Explanation,Sample,Stability Score,Stability Std,Valid Variations
0,AnoFormer,Attention,Normal,0.999992,1.170278e-16,10
1,AnoFormer,Attention,Anomalous,0.999967,0.000000e+00,10


In [19]:
# Save the Results
anoformer_attention_stability_path = STABILITY_DIR / "anoformer_attention_stability_results.csv"

AnoFormerAttentionStabilityResults.to_csv(
    anoformer_attention_stability_path,
    index=False,
)

print("Saved to:", anoformer_attention_stability_path)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/stability/anoformer_attention_stability_results.csv


## SHAP Evaluation

In [12]:
# Load the background indices used for SHAP explanations

background_indices = np.load(
    SHAP_RESULTS_DIR / "medium_background_indices.npy"
)

background_ECGs = torch.as_tensor(
    X_train[background_indices],
    dtype=torch.float32,
    device=DEVICE,
)

In [21]:
# Create the same Anomaly Scpre Wrapper for AnoFormer that was used during SHAP explanations

ScoreWrapper = AnoFormerAnomalyScoreWrapper(
    anoformer_model=AnoFormer,
    window_size=1000,
    masking_rate=None,
    num_masks=1,
    seed=RANDOM_SEED,
).to(DEVICE)

ScoreWrapper.eval()

AnoFormerAnomalyScoreWrapper(
  (generator): TransformerGenerator(
    (InputEmbedding): InputEmbeddingLayer(
      (ProjectionLayer): Linear(in_features=12, out_features=64, bias=True)
      (ActivationReLU): ReLU()
    )
    (PositionalEncoding): PositionalEncoding(
      (DropoutLayer): Dropout(p=0.1, inplace=False)
    )
    (EncoderStack): TransformerEncoderStack(
      (EncoderBlockList): ModuleList(
        (0-1): 2 x TransformerEncoderBlock(
          (SelfAttention): MultiHeadAttention(
            (LinearLayerQuery): Linear(in_features=64, out_features=64, bias=False)
            (LinearLayerKey): Linear(in_features=64, out_features=64, bias=False)
            (LinearLayerValue): Linear(in_features=64, out_features=64, bias=False)
            (LinearLayerOutput): Linear(in_features=64, out_features=64, bias=True)
            (AttentionDropoutLayer): Dropout(p=0.1, inplace=False)
          )
          (NormalizationOne): LayerNormalization()
          (ForwardNetworkLayer): Fo

In [22]:
# Create the same Explainer

GradientSHAPExplainer = shap.GradientExplainer(
    ScoreWrapper,
    background_ECGs,
)

In [13]:
def get_temporal_shap_importance_from_values(shap_values):
    """
    Convert SHAP values to timestep-level temporal importance.

    Expected SHAP shape after removing output dimension:
    - (1, 1000, 12), or
    - (1000, 12)

    Output:
    - (1000,)
    """

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)

    # Remove output dimension if shape is (1, 1000, 12, 1)
    if shap_values.ndim == 4 and shap_values.shape[-1] == 1:
        shap_values = shap_values[..., 0]

    # Remove batch dimension if shape is (1, 1000, 12)
    shap_values = np.squeeze(shap_values)

    if shap_values.shape != (1000, 12):
        raise ValueError(
            f"Expected SHAP values with shape (1000, 12), "
            f"but got {shap_values.shape}."
        )

    # Sum absolute SHAP values over the 12 leads
    temporal_importance = np.abs(shap_values).sum(axis=1)

    return temporal_importance

In [14]:
def shap_importance_function(ecg):
    """
    Recalculate SHAP values for one ECG and return timestep-level
    temporal SHAP importance with shape (1000,).
    """

    if isinstance(ecg, np.ndarray):
        ecg = torch.tensor(ecg, dtype=torch.float32)

    if ecg.ndim == 2:
        ecg = ecg.unsqueeze(0)

    ecg = ecg.to(DEVICE)

    shap_values = GradientSHAPExplainer.shap_values(ecg)

    temporal_importance = get_temporal_shap_importance_from_values(
        shap_values
    )

    return temporal_importance

In [25]:
# Calculate SHAP values for original ECGs

NormalSHAPImportance = shap_importance_function(NormalECG)
AnomalousSHAPImportance = shap_importance_function(AnomalousECG)

print("Normal SHAP importance:", NormalSHAPImportance.shape)
print("Anomalous SHAP importance:", AnomalousSHAPImportance.shape)

Normal SHAP importance: (1000,)
Anomalous SHAP importance: (1000,)


In [26]:
# Generate valid noisy variants of the normal ECG and collect their SHAP importance values and scores

NormalNoisySHAPImportances, NormalNoisySHAPScores = collect_valid_noisy_importances(
    ecg=NormalECG,
    score_function=anoformer_score_function,
    explanation_function=shap_importance_function,
    threshold=AnoFormerChosenThreshold,
    stability_evaluator=StabilityEvaluator,
    number_variations=10,
)

AnomalousNoisySHAPImportances, AnomalousNoisySHAPScores = collect_valid_noisy_importances(
    ecg=AnomalousECG,
    score_function=anoformer_score_function,
    explanation_function=shap_importance_function,
    threshold=AnoFormerChosenThreshold,
    stability_evaluator=StabilityEvaluator,
    number_variations=10,
)

In [27]:
# Evaluate AnoFormer SHAP stability

AnoFormerSHAPNormalStability = StabilityEvaluator.evaluate(
    original_importance=NormalSHAPImportance,
    noisy_importance_values=NormalNoisySHAPImportances,
)

AnoFormerSHAPAnomalousStability = StabilityEvaluator.evaluate(
    original_importance=AnomalousSHAPImportance,
    noisy_importance_values=AnomalousNoisySHAPImportances,
)

print("Normal SHAP stability:", AnoFormerSHAPNormalStability)
print("Anomalous SHAP stability:", AnoFormerSHAPAnomalousStability)

Normal SHAP stability: {'stability_score': 0.9999890739684643, 'stability_std': 0.0, 'cosine_similarities': array([0.99998907, 0.99998907, 0.99998907, 0.99998907, 0.99998907,
       0.99998907, 0.99998907, 0.99998907, 0.99998907, 0.99998907])}
Anomalous SHAP stability: {'stability_score': 0.9988135394519218, 'stability_std': 0.0, 'cosine_similarities': array([0.99881354, 0.99881354, 0.99881354, 0.99881354, 0.99881354,
       0.99881354, 0.99881354, 0.99881354, 0.99881354, 0.99881354])}


In [28]:
# Print the Results

AnoFormerSHAPStabilityResults = pd.DataFrame([
    {
        "Model": "AnoFormer",
        "Explanation": "SHAP",
        "Sample": "Normal",
        "Stability Score": AnoFormerSHAPNormalStability["stability_score"],
        "Stability Std": AnoFormerSHAPNormalStability["stability_std"],
        "Valid Variations": len(NormalNoisySHAPImportances),
    },
    {
        "Model": "AnoFormer",
        "Explanation": "SHAP",
        "Sample": "Anomalous",
        "Stability Score": AnoFormerSHAPAnomalousStability["stability_score"],
        "Stability Std": AnoFormerSHAPAnomalousStability["stability_std"],
        "Valid Variations": len(AnomalousNoisySHAPImportances),
    },
])

AnoFormerSHAPStabilityResults

,Model,Explanation,Sample,Stability Score,Stability Std,Valid Variations
0,AnoFormer,SHAP,Normal,0.999989,0.0,10
1,AnoFormer,SHAP,Anomalous,0.998814,0.0,10


In [29]:
# Save the Results

anoformer_shap_stability_path = STABILITY_DIR / "anoformer_shap_stability_results.csv"

AnoFormerSHAPStabilityResults.to_csv(
    anoformer_shap_stability_path,
    index=False,
)

print("Saved to:", anoformer_shap_stability_path)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/stability/anoformer_shap_stability_results.csv


# TranAD Evaluation

In [ ]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)


TranADModel: 268,376 Parameter | d_model=64, H=4, Enc=2, Dec×2=2
Model loaded  /content/drive/MyDrive/ECG data/results/models/tranad/tranad_20epochs_larger_capacity.pt


In [20]:
def tranad_score_function(ecg):
    """
    Return the TranAD anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = TranAD.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=TranADChosenThreshold,
    )

    return scores[0]

## Attention Evaluation

In [32]:
# A helper function to get the attention importance values for a given ECG
# This is needed because we neeed to pass multiple ecgs

def tranad_attention_importance_function(ecg):
    return TranADExplainer.GetAttentionImportance(
        TimeSeriesWindow=NormalECG,
        LayerIndex=0,
        HeadIndex=None,
    )

In [33]:
# Extract the original attention importance values for the normal and anomalous ECGs

NormalAttentionImportance = tranad_attention_importance_function(NormalECG)
AnomalousAttentionImportance = tranad_attention_importance_function(AnomalousECG)

In [34]:
# Generate valid noisy variants of the normal ECG and collect their attention importance values and scores

NormalNoisyAttentionImportances, NormalNoisyScores = collect_valid_noisy_importances(
    ecg=NormalECG,
    score_function=tranad_score_function,
    explanation_function=tranad_attention_importance_function,
    threshold=TranADChosenThreshold,
    stability_evaluator=StabilityEvaluator,
    number_variations=10,
)

AnomalousNoisyAttentionImportances, AnomalousNoisyScores = collect_valid_noisy_importances(
    ecg=AnomalousECG,
    score_function=tranad_score_function,
    explanation_function=tranad_attention_importance_function,
    threshold=TranADChosenThreshold,
    stability_evaluator=StabilityEvaluator,
    number_variations=10,
)

In [35]:
# Evaluate
TranADAttentionNormalStability = StabilityEvaluator.evaluate(
    original_importance=NormalAttentionImportance,
    noisy_importance_values=NormalNoisyAttentionImportances,
)

TranADAttentionAnomalousStability = StabilityEvaluator.evaluate(
    original_importance=AnomalousAttentionImportance,
    noisy_importance_values=AnomalousNoisyAttentionImportances,
)

In [36]:
# Print the Results
TranADAttentionStabilityResults = pd.DataFrame([
    {
        "Model": "TranAD",
        "Explanation": "Attention",
        "Sample": "Normal",
        "Stability Score": TranADAttentionNormalStability["stability_score"],
        "Stability Std": TranADAttentionNormalStability["stability_std"],
        "Valid Variations": len(NormalNoisyAttentionImportances),
    },
    {
        "Model": "TranAD",
        "Explanation": "Attention",
        "Sample": "Anomalous",
        "Stability Score": TranADAttentionAnomalousStability["stability_score"],
        "Stability Std": TranADAttentionAnomalousStability["stability_std"],
        "Valid Variations": len(AnomalousNoisyAttentionImportances),
    },
])

TranADAttentionStabilityResults

,Model,Explanation,Sample,Stability Score,Stability Std,Valid Variations
0,TranAD,Attention,Normal,0.999993,0.0,10
1,TranAD,Attention,Anomalous,0.999993,0.0,10


In [37]:
# Save the Results
tranad_attention_stability_path = STABILITY_DIR / "tranad_attention_stability_results.csv"

TranADAttentionStabilityResults.to_csv(
    tranad_attention_stability_path,
    index=False,
)

print("Saved to:", tranad_attention_stability_path)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/stability/tranad_attention_stability_results.csv


## SHAP Evaluation

In [16]:
# Create the same TranAD Anomaly Score Wrapper for TranAD that was used during SHAP explanations

TranADScoreWrapper = TranADAnomalyScoreWrapper(TranAD)

In [17]:
# Create the same Explainer

GradientSHAPExplainer = shap.GradientExplainer(
    TranADScoreWrapper,
    background_ECGs,
)

In [18]:
# Calculate SHAP values for original ECGs

TranADNormalSHAPImportance = shap_importance_function(NormalECG)
TranADAnomalousSHAPImportance = shap_importance_function(AnomalousECG)

print("Normal SHAP importance:", TranADNormalSHAPImportance.shape)
print("Anomalous SHAP importance:", TranADAnomalousSHAPImportance.shape)

Normal SHAP importance: (1000,)
Anomalous SHAP importance: (1000,)


In [21]:
# Generate valid noisy variants of the normal ECG and collect their SHAP importance values and scores

TranADNormalNoisySHAPImportances, TranADNormalNoisySHAPScores = collect_valid_noisy_importances(
    ecg=NormalECG,
    score_function=tranad_score_function,
    explanation_function=shap_importance_function,
    threshold=TranADChosenThreshold,
    stability_evaluator=StabilityEvaluator,
    number_variations=10,
)

TranADAnomalousNoisySHAPImportances, TranADAnomalousNoisySHAPScores = collect_valid_noisy_importances(
    ecg=AnomalousECG,
    score_function=tranad_score_function,
    explanation_function=shap_importance_function,
    threshold=TranADChosenThreshold,
    stability_evaluator=StabilityEvaluator,
    number_variations=10,
)

In [22]:
# Evaluate TranAD SHAP stability

TranADSHAPNormalStability = StabilityEvaluator.evaluate(
    original_importance=TranADNormalSHAPImportance,
    noisy_importance_values=TranADNormalNoisySHAPImportances,
)

TranADSHAPAnomalousStability = StabilityEvaluator.evaluate(
    original_importance=TranADAnomalousSHAPImportance,
    noisy_importance_values=TranADAnomalousNoisySHAPImportances,
)

print("Normal SHAP stability:", TranADSHAPNormalStability)
print("Anomalous SHAP stability:", TranADSHAPAnomalousStability)

Normal SHAP stability: {'stability_score': 0.9988454490247374, 'stability_std': 0.0002701035841413138, 'cosine_similarities': array([0.9986451 , 0.99901084, 0.99884729, 0.99890057, 0.99912701,
       0.99825285, 0.99904476, 0.99862811, 0.99889527, 0.99910269])}
Anomalous SHAP stability: {'stability_score': 0.9960254397970679, 'stability_std': 0.0044545363466071495, 'cosine_similarities': array([0.99961111, 0.98591773, 0.99679531, 0.98994185, 0.9980029 ,
       0.99774696, 0.99842384, 0.99866046, 0.99846466, 0.99668958])}


In [23]:
# Print the Results

TranADSHAPStabilityResults = pd.DataFrame([
    {
        "Model": "TranAD",
        "Explanation": "SHAP",
        "Sample": "Normal",
        "Stability Score": TranADSHAPNormalStability["stability_score"],
        "Stability Std": TranADSHAPNormalStability["stability_std"],
        "Valid Variations": len(TranADNormalNoisySHAPImportances),
    },
    {
        "Model": "TranAD",
        "Explanation": "SHAP",
        "Sample": "Anomalous",
        "Stability Score": TranADSHAPAnomalousStability["stability_score"],
        "Stability Std": TranADSHAPAnomalousStability["stability_std"],
        "Valid Variations": len(TranADAnomalousNoisySHAPImportances),
    },
])

TranADSHAPStabilityResults

,Model,Explanation,Sample,Stability Score,Stability Std,Valid Variations
0,TranAD,SHAP,Normal,0.998845,0.000270,10
1,TranAD,SHAP,Anomalous,0.996025,0.004455,10


In [24]:
# Save the Results

tranad_shap_stability_path = STABILITY_DIR / "tranad_shap_stability_results.csv"

TranADSHAPStabilityResults.to_csv(
    tranad_shap_stability_path,
    index=False,
)

print("Saved to:", tranad_shap_stability_path)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/stability/tranad_shap_stability_results.csv
